In [ ]:
import os
import time
import joblib
import numpy as np
import polars as pl
import psutil

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

# OPTUNA
import optuna

# XGBOOST
from xgboost import XGBClassifier

In [ ]:
# ==========================================
# 1. CARGA DE DATOS
# ==========================================

path_train = "../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test  = "../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"

df_train = pl.read_csv(path_train)
df_test  = pl.read_csv(path_test)

TARGET_COL = "attack_cat"

# 2. Crear y_train e y_test (Normal=1, Ataque=-1)
y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal")
        .then(1)
        .otherwise(-1)
        .alias("label")
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test  = df_test.drop(TARGET_COL)

print(f"Forma de x_train: {x_train.shape} | Clases únicas en y_train: {y_train.unique().to_list()}")
print("\nDistribución de clases en Train:")
print(y_train.value_counts())
print("\nDistribución de clases en Test:")
print(y_test.value_counts())

# Convertimos a NumPy (como RF)
X_full_train = x_train.to_numpy()
y_full_train = y_train.to_numpy()

# Split interno (no imprescindible para CV, pero lo mantenemos como lo tenías)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_full_train, y_full_train, test_size=0.2, random_state=42, stratify=y_full_train
)

X_test_np = x_test.to_numpy()
y_test_np = y_test.to_numpy()

print(f"Entrenamiento: {X_train_np.shape[0]} muestras")
print(f"Validación:    {X_val_np.shape[0]} muestras")
print(f"Test:          {X_test_np.shape[0]} muestras")

In [ ]:
# ==========================================
# FASE 1 - OPTUNA + 3-FOLD CV (XGBOOST)
# Objetivos: max F1_macro, min latencia
# Hiperparámetros estructurales: n_estimators, max_depth
# ==========================================

def objective(trial):
    # 1) Sugerir hiperparámetros estructurales
    n_estimators = trial.suggest_int("n_estimators", 50, 600, step=50)
    max_depth = trial.suggest_int("max_depth", 2, 12)  # en XGB >12 suele ser excesivo

    # 2) Configurar CV
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    f1_scores = []
    latencies = []

    for fold_id, (train_idx, val_idx) in enumerate(skf.split(X_full_train, y_full_train), start=1):
        X_train_cv, X_val_cv = X_full_train[train_idx], X_full_train[val_idx]
        y_train_cv, y_val_cv = y_full_train[train_idx], y_full_train[val_idx]

        # XGBoost espera etiquetas 0/1 idealmente para binary:logistic.
        # Convertimos -1/1 -> 0/1 localmente en cada fold:
        y_train_cv01 = ((y_train_cv + 1) // 2).astype(np.int8)
        y_val_cv01   = ((y_val_cv + 1) // 2).astype(np.int8)

        # 3) Modelo XGBoost
        model = XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,

            # parámetros "seguros" por defecto para un baseline estable
            learning_rate=0.1,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1.0,
            reg_lambda=1.0,

            objective="binary:logistic",
            eval_metric="logloss",

            n_jobs=-1,
            random_state=42,
            tree_method="hist"  # suele acelerar en CPU
        )

        # 4) Entrenamiento
        model.fit(X_train_cv, y_train_cv01)

        # 5) Eficacia: F1 macro
        # predict() devuelve 0/1
        y_pred01 = model.predict(X_val_cv)
        f1_scores.append(f1_score(y_val_cv01, y_pred01, average="macro"))

        # 6) Eficiencia: latencia
        # Warm-up
        warm_n = min(1000, len(X_val_cv))
        _ = model.predict(X_val_cv[:warm_n])

        # Medimos sobre subset fijo para que no tarde mucho y sea comparable
        subset = min(20000, len(X_val_cv))
        X_lat = X_val_cv[:subset]

        # Repetimos 3 veces (puedes subir a 5 si quieres más estabilidad)
        rep = 3
        fold_lat = []
        for _ in range(rep):
            t0 = time.perf_counter()
            _ = model.predict(X_lat)
            t1 = time.perf_counter()
            fold_lat.append((t1 - t0) / len(X_lat) * 1000)

        latencies.append(float(np.mean(fold_lat)))

    avg_f1 = float(np.mean(f1_scores))
    avg_lat = float(np.mean(latencies))
    std_f1 = float(np.std(f1_scores))

    trial.set_user_attr("f1_std", std_f1)

    return avg_f1, avg_lat


study = optuna.create_study(
    directions=["maximize", "minimize"],
    study_name="tfg_ids_xgb_optimization_cv"
)

print("🚀 Iniciando barrido multiobjetivo con XGBoost + 3-Fold CV...")
print("Nota: Cada trial entrena 3 modelos. Esto será más lento pero más robusto.")

study.optimize(objective, n_trials=50)


# ==========================================
# GUARDAR RESULTADOS A CSV (POLARS)
# ==========================================

# Ojo: mejor marcar Pareto por number (más robusto que 't in best_trials')
pareto_ids = {t.number for t in study.best_trials}

trials_data = []
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        trials_data.append({
            "n_estimators": t.params["n_estimators"],
            "max_depth": t.params["max_depth"],
            "f1_macro": t.values[0],
            "f1_std": t.user_attrs["f1_std"],
            "latency_ms": t.values[1],
            "is_pareto": t.number in pareto_ids
        })

df_results = pl.DataFrame(trials_data)
df_results.write_csv("xgb_trials_results_cv.csv")

print("\n✅ Resultados robustos guardados en 'xgb_trials_results_cv.csv'")
print(df_results.sort("f1_macro", descending=True).head())

In [ ]:
# ==========================================
# GRAFICA PARETO (Polars + NumPy Edition)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

df = pl.read_csv("xgb_trials_results_cv.csv")

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

sns.scatterplot(
    x=df["latency_ms"].to_numpy(),
    y=df["f1_macro"].to_numpy(),
    hue=df["is_pareto"].to_numpy(),
    palette={True: "red", False: "royalblue"},
    style=df["is_pareto"].to_numpy(),
    markers={True: "D", False: "o"},
    s=100,
    alpha=0.8
)

pareto_points = df.filter(pl.col("is_pareto") == True)

for row in pareto_points.iter_rows(named=True):
    plt.text(
        row["latency_ms"],
        row["f1_macro"] + 0.0005,
        f"n={int(row['n_estimators'])}, d={int(row['max_depth'])}",
        fontsize=9, fontweight='bold', ha='center'
    )

plt.title("Análisis Multiobjetivo (XGBoost): Eficacia (F1) vs. Latencia", fontsize=15)
plt.xlabel("Latencia (ms/muestra)", fontsize=12)
plt.ylabel("F1-Score Macro", fontsize=12)
plt.legend(title="¿Es Pareto?")

plt.show()

In [ ]:
# ==========================================
# EVALUACIÓN FINAL EN TEST (3 CANDIDATOS)
# (rellenas candidatos según tus Pareto)
# ==========================================

candidatos = [] # Los ajustamos cuando veamos el pareto

resultados_finales = []

print("--- EVALUACIÓN FINAL SOBRE EL SET DE TEST (XGBoost) ---\n")

# Convertimos y_train/y_test a 0/1 para XGB
y_full_train01 = ((y_full_train + 1) // 2).astype(np.int8)
y_test_np01    = ((y_test_np + 1) // 2).astype(np.int8)

for c in candidatos:
    print(f"Probando: {c['nombre']} (n={c['n']}, d={c['d']})...")

    model = XGBClassifier(
        n_estimators=c["n"],
        max_depth=c["d"],
        learning_rate=0.1,
        subsample=1.0,
        colsample_bytree=1.0,
        min_child_weight=1.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42,
        tree_method="hist"
    )

    model.fit(X_full_train, y_full_train01)

    # Warm-up
    _ = model.predict(X_test_np[:min(1000, len(X_test_np))])

    t0 = time.perf_counter()
    y_pred01 = model.predict(X_test_np)
    t1 = time.perf_counter()

    tiempo_total = t1 - t0
    latencia = (tiempo_total / len(y_test_np01)) * 1000
    f1_test = f1_score(y_test_np01, y_pred01, average="macro")
    acc_test = accuracy_score(y_test_np01, y_pred01)

    resultados_finales.append({
        "Perfil": c["nombre"],
        "n_estimators": c["n"],
        "max_depth": c["d"],
        "F1_Test": float(f1_test),
        "Accuracy_Test": float(acc_test),
        "Latencia_ms": float(latencia)
    })

df_final = pl.DataFrame(resultados_finales)
print("\n" + "="*60)
print("              TABLA COMPARATIVA FINAL (XGBoost)")
print("="*60)
print(df_final)